# Sql Intermedio

## Chinook Database utilizado

### Detalle de database

- **artists**: Nombre de artistas musicales.
- **albums**: Álbumes, cada uno ligado a un artista.
- **tracks**: Canciones individuales, cada una ligada a un álbum.
- **genres**: Géneros musicales (rock, jazz, metal, etc.).
- **media_types**: Tipos de archivo (MPEG audio, AAC, etc.).
- **playlists**: Listas de reproducción.
- **playlist_track**: Tabla intermedia que conecta playlists con tracks (relación muchos-a-muchos).
- **customers**: Datos de clientes.
- **employees**	Datos de empleados, con jerarquía (ReportsTo).
- **invoices**:	Cabecera de facturas (cliente, fecha, total).
- **invoice_items**: Líneas de detalle de cada factura (qué track se compró, precio, cantidad).

### Modelo de datos

<img src="imagenes/Diagrama_entidad_relacion_Chinook.png" width="600">

### Descarga y exploración de la base de datos

- **Descarga del archivo:** `urllib.request.urlretrieve(url, nombre_local)` descarga la base de datos Chinook (SQLite) desde GitHub y la guarda localmente — a diferencia del dataset de retail, esta base ya viene completa con datos, no se carga desde un CSV.
- **Conexión:** `sqlite3.connect('data/sql/Chinook_Sqlite.sqlite')` abre la conexión al archivo descargado, igual que con cualquier base SQLite.
- **Listar tablas existentes:** la consulta `SELECT name FROM sqlite_master WHERE type='table';` consulta `sqlite_master`, una tabla interna que SQLite mantiene automáticamente con los metadatos de todos los objetos de la base (tablas, índices, vistas). Filtrar por `type='table'` muestra solo los nombres de las tablas reales disponibles (Artist, Album, Track, Customer, Invoice, etc.).
- **Por qué es útil:** es la forma de verificar qué tablas existen realmente dentro de un archivo `.sqlite`, sin adivinar los nombres de memoria — la misma consulta de diagnóstico que resuelve errores tipo `no such table`.
- **Por qué Chinook:** a diferencia del dataset de retail (una sola tabla plana), Chinook tiene múltiples tablas relacionadas entre sí, ideal para practicar `JOIN` y SQL intermedio.

In [2]:
import sqlite3
import pandas as pd
import urllib.request

url = 'https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite'
urllib.request.urlretrieve(url, 'data/sql/Chinook_Sqlite.sqlite')

conn = sqlite3.connect('data/sql/Chinook_Sqlite.sqlite')

tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(tables_query, conn)
tables

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


**Requisitos principales para utilizar magic:** 
- !pip install jupysql

El comando "%load_ext sql" inicializa la extencion "magic" de sql en el entorno jupyter, y permite ejecutar consultas SQL dentro del codigo.

El comando "%sql sqlite:///data/sql/Chinook_Sqlite.sqlite" crea una conexion con nuetra base de datos local de SQLite.

In [3]:
%load_ext sql

%sql sqlite:///data/sql/Chinook_Sqlite.sqlite

Connecting to 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

## Funciones de agregacion

###  Count
Es una funcion para contar el numero de filas en una columna especifica. Una de las ventajas que tiene es que se puede utilizar en columnas no numericas.

Cuenta los valores nulos.

In [4]:
%%sql
SELECT COUNT('Name') as 'Conteo tabla Artist'
FROM "Artist"

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Conteo tabla Artist
275


### Sum
Suma los valores de una columna determinada, solo se puede utilizar en columnas numericas.

Los valores nulos los trata como 0.

In [5]:
%%sql
SELECT SUM("UnitPrice") as 'Suma Unitprice'
FROM "InvoiceLine"

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Suma Unitprice
2328.6


### Min/Max

- Min: Devueve el valor minimo de la columna.
- Max: Devueve el valor maximo de la columna.

Se pueden utilizar en columnas no numericas:
- Min: Puede devolver la fecha mas antigua, tambien el valor no numerico mas cercano alfabeticamente a "A".
- Max: Puede devolver la fecha mas reciente, tambien el valor no numerico mas cercano alfabeticamente a "Z".

In [6]:
%%sql
-- Devuelve una tabla con: nombre de columna, tipo de dato, si permite nulos, valor por defecto, y si es clave primaria.
PRAGMA table_info("Invoice") 

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

cid,name,type,notnull,dflt_value,pk
0,InvoiceId,INTEGER,1,None,1
1,CustomerId,INTEGER,1,None,0
2,InvoiceDate,DATETIME,1,None,0
3,BillingAddress,NVARCHAR(70),0,None,0
4,BillingCity,NVARCHAR(40),0,None,0
5,BillingState,NVARCHAR(40),0,None,0
6,BillingCountry,NVARCHAR(40),0,None,0
7,BillingPostalCode,NVARCHAR(10),0,None,0
8,Total,"NUMERIC(10,2)",1,None,0


In [7]:
%%sql
SELECT 
MIN("Total") as 'Min Total',
MAx("Total") as 'Max Total'
FROM "Invoice"

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Min Total,Max Total
0.99,25.86


In [8]:
%%sql
SELECT 
MIN("InvoiceDate") as 'Min InvoiceDate',
MAx("InvoiceDate") as 'Max InvoiceDate'
FROM "Invoice"

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Min InvoiceDate,Max InvoiceDate
2021-01-01 00:00:00,2025-12-22 00:00:00


In [9]:
%%sql
SELECT 
MIN("BillingCity") as 'Min BillingCity',
MAx("BillingCity") as 'Max BillingCity'
FROM "Invoice"

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Min BillingCity,Max BillingCity
Amsterdam,Yellowknife


### Avg
Calcula el promedio de un grupo seleccionado de valores. Tiene ciertas limitaciones entre ellas:
- Solo se puede utilizar en columnas numericas.
- Ignora los valores nulos.

In [10]:
%%sql
PRAGMA table_info("Track") 

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

cid,name,type,notnull,dflt_value,pk
0,TrackId,INTEGER,1,None,1
1,Name,NVARCHAR(200),1,None,0
2,AlbumId,INTEGER,0,None,0
3,MediaTypeId,INTEGER,1,None,0
4,GenreId,INTEGER,0,None,0
5,Composer,NVARCHAR(220),0,None,0
6,Milliseconds,INTEGER,1,None,0
7,Bytes,INTEGER,0,None,0
8,UnitPrice,"NUMERIC(10,2)",1,None,0


In [11]:
%%sql 
SELECT AVG("Milliseconds") as 'Promedio Milliseconds'
FROM "Track"

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Promedio Milliseconds
393599.2121039109


### Group By
Permite separar datos en grupos, que pueden agregarse independientemente unos de otros. Se pueden agrupar por varias columnas, estas se deben separar con una coma.

In [12]:
%%sql
SELECT BillingCity, COUNT(*) as Count
FROM "Invoice"
GROUP BY BillingCity

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingCity,Count
Amsterdam,7
Bangalore,6
Berlin,14
Bordeaux,7
Boston,7
Brasília,7
Brussels,7
Budapest,7
Buenos Aires,7
Chicago,7


In [13]:
%%sql
SELECT BillingState, BillingCity, COUNT(*) as Count
FROM "Invoice"
GROUP BY BillingState, BillingCity
LIMIT 20

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingState,BillingCity,Count
None,Bangalore,6
None,Berlin,14
None,Bordeaux,7
None,Brussels,7
None,Budapest,7
None,Buenos Aires,7
None,Copenhagen,7
None,Delhi,7
None,Dijon,7
None,Edinburgh,7


#### AGRUPAR POR números de columna
Se pueden sustituir los nombres de las columas por numeros. Generalmente se recomienda hacerlo solo cuando se agruoan mucas columnas.

In [14]:
%%sql
SELECT BillingState, BillingCity, COUNT(*) as Count
FROM "Invoice"
GROUP BY 1, 2

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingState,BillingCity,Count
None,Bangalore,6
None,Berlin,14
None,Bordeaux,7
None,Brussels,7
None,Budapest,7
None,Buenos Aires,7
None,Copenhagen,7
None,Delhi,7
None,Dijon,7
None,Edinburgh,7


#### GROUP BY con ORDER BY
Controla el orden en el que se agrupan las agregaciones.

In [15]:
%%sql
SELECT BillingState, BillingCity, COUNT(*) as Count
FROM "Invoice"
GROUP BY 1, 2
order by 2, 1

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingState,BillingCity,Count
VV,Amsterdam,7
None,Bangalore,6
None,Berlin,14
None,Bordeaux,7
MA,Boston,7
DF,Brasília,7
None,Brussels,7
None,Budapest,7
None,Buenos Aires,7
IL,Chicago,7


### Having
Es para filtrar como la clausula "WHERE".

In [16]:
%%sql
SELECT BillingState, BillingCity, MAX(Total) as Max_Total
FROM "Invoice"
GROUP BY 1, 2
HAVING MAX(Total) > 25
order by 2, 1

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingState,BillingCity,Max_Total
None,Prague,25.86


### Orden de la cláusula de consulta: 

1. SELECT
2. FROM
3. WHERE
4. GROUP BY
5. HAVING
6. ORDER BY

### Case
Maneja la logica de if/then. La instruccion va seguida de varios "WHEN" y instrucciones "THEN". Cada instruccion debe terminar en un "END".

In [17]:
%%sql
select BillingCountry from Invoice

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingCountry
Germany
Norway
Belgium
Canada
USA
Germany
Germany
France
France
Ireland


In [18]:
%%sql

SELECT BillingCountry, InvoiceDate,
CASE 
    WHEN BillingCountry = 'Norway' THEN 'NW'
    WHEN BillingCountry = "Germany" THEN 'GM'
    ELSE NULL 
END AS 'Tabla'
FROM Invoice

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingCountry,InvoiceDate,Tabla
Germany,2021-01-01 00:00:00,GM
Norway,2021-01-02 00:00:00,NW
Belgium,2021-01-03 00:00:00,None
Canada,2021-01-06 00:00:00,None
USA,2021-01-11 00:00:00,None
Germany,2021-01-19 00:00:00,GM
Germany,2021-02-01 00:00:00,GM
France,2021-02-01 00:00:00,None
France,2021-02-02 00:00:00,None
Ireland,2021-02-03 00:00:00,None


### Distinct
Sirve oara ver los valores unicos en una columna.

In [19]:
%%sql

SELECT DISTINCT BillingCountry
FROM Invoice

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingCountry
Germany
Norway
Belgium
Canada
USA
France
Ireland
United Kingdom
Australia
Chile


Si se utiliza con 2 o mas columnas, sus resultados tendran todos los pares unicos de esos columnas.

In [20]:
%%sql

SELECT DISTINCT BillingCountry, BillingState
FROM Invoice

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

BillingCountry,BillingState
Germany,None
Norway,None
Belgium,None
Canada,AB
USA,MA
France,None
Ireland,Dublin
United Kingdom,None
USA,CA
USA,WA


#### DISTINCT en agregaciones
Se puede usar DISTINCT al realizar una agregación. Usado con mayor frecuencia con la COUNT función.

In [21]:
%%sql

SELECT COUNT(DISTINCT InvoiceDate) as 'Fechas Distintas'
FROM Invoice

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Fechas Distintas
354


### Joins
Sirve para unir tablas que tengan un dato en comun y obtener informacion complementaria de la tablas que se unen.

Por eso lo mejor es tener a la mano el modelo de datos para ver que tablas se pueden unir y los atributos que pueden compartir.

#### Modelo de datos

<img src="imagenes/Diagrama_entidad_relacion_Chinook.png" width="600">

En el siguiente ejemplo se obtiene informacion de la tabla "Track" y "Album", para obtener informacion de el album y cada track que que tiene el album.

In [40]:
%%sql

SELECT 
    a.Title AS "Titulo Album",
    t.Name AS "Nombre Cancion"
FROM Track t 
Join Album a ON a.AlbumId = t.AlbumId 
ORDER BY a.title

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Titulo Album,Nombre Cancion
...And Justice For All,Blackened
...And Justice For All,...And Justice For All
...And Justice For All,Eye Of The Beholder
...And Justice For All,One
...And Justice For All,The Shortest Straw
...And Justice For All,Harvester Of Sorrow
...And Justice For All,The Frayed Ends Of Sanity
...And Justice For All,To Live Is To Die
...And Justice For All,Dyers Eve
20th Century Masters - The Millennium Collection: The Best of Scorpions,Rock You Like a Hurricane


#### Join Multiple

Tambien se puede obtener el artista uniendo las 3 tablas aunque no tengan algo en comun las 3, mientras una de ellas tenga una forma de unirla, la tabla "Artist" tiene ArtistId que se puede unir con "Album" ArtistID.

In [41]:
%%sql

SELECT 
    artist.Name AS "Nombre Artista",
    album.Title AS "Titulo Album",
    track.Name AS "Nombre Cancion"
    
FROM Track AS track 
JOIN Album AS album ON album.AlbumId = track.AlbumId
JOIN Artist AS artist ON artist.ArtistId = album.ArtistId
ORDER BY artist.name, album.Title


Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Nombre Artista,Titulo Album,Nombre Cancion
AC/DC,For Those About To Rock We Salute You,For Those About To Rock (We Salute You)
AC/DC,For Those About To Rock We Salute You,Put The Finger On You
AC/DC,For Those About To Rock We Salute You,Let's Get It Up
AC/DC,For Those About To Rock We Salute You,Inject The Venom
AC/DC,For Those About To Rock We Salute You,Snowballed
AC/DC,For Those About To Rock We Salute You,Evil Walks
AC/DC,For Those About To Rock We Salute You,C.O.D.
AC/DC,For Those About To Rock We Salute You,Breaking The Rules
AC/DC,For Those About To Rock We Salute You,Night Of The Long Knives
AC/DC,For Those About To Rock We Salute You,Spellbound


#### Inner Join

Combina filas de dos tablas, mostrando solo las combinaciones donde hay coincidencias en ambas tablas. Si una fila de ina tabla no tiene una fila correpondiente en la otra, se excluye del resultado.

Como en el mismo caso anterior ya que al escribir solamente "JOIN" se entiende que es inner join y que solo apareceran los datos que tengan un "AlbumId" que sea igual en ambas tablas.

In [43]:
%%sql

SELECT 
    a.Title AS "Titulo Album",
    t.Name AS "Nombre Cancion"
FROM Track t 
Join Album a ON a.AlbumId = t.AlbumId 
ORDER BY a.title
LIMIT 4

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Titulo Album,Nombre Cancion
...And Justice For All,Blackened
...And Justice For All,...And Justice For All
...And Justice For All,Eye Of The Beholder
...And Justice For All,One


#### Outer Join
Son uniones que devuelven valores coincidentes y valores no coincidentes de una o ambas tablas. Existen varios tipos:
- **LEFT JOIN**: Tambien conocida como "OUTER LEFT JOIN". Devuelve unicamente las filas no coincidentes de la tabla izquierda, asi como las filas coincidentes de ambas tablas.
- **RIGHT JOIN**: Tambien conocida como "OUTER RIGHT JOIN". Devuelve unicamente las filas no coincidentes de la tabla derecha, asi como las filas coincidentes de ambas tablas.
- **FULL OUTER JOIN**: Tambien conocida como "OUTER JOIN". Devuelve las filas no coincidentes de ambas tablas, aso como las filas coincidentes de ambas tablas.

<img src="imagenes/visual-join.png" width="600">

##### Left Join
El siguiente ejemplo trae todos los artistas de la tabla "Artist", tengan o no un álbum asociado. Si un artista no tiene ningún álbum en la tabla "Album", igual aparece en el resultado, pero con None en la columna Album.

In [46]:
%%sql
SELECT 
    ar.Name AS Artista,
    a.Title AS Album
FROM Artist AS ar
LEFT JOIN Album AS a ON a.ArtistId = ar.ArtistId
ORDER BY ar.Name
LIMIT 5

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Artista,Album
A Cor Do Som,None
AC/DC,For Those About To Rock We Salute You
AC/DC,Let There Be Rock
Aaron Copland & London Symphony Orchestra,"A Copland Celebration, Vol. I"
Aaron Goldberg,Worlds


##### Right Join
El siguiente ejemplo trae todos los álbumes de la tabla "Album", tengan o no un artista válido asociado. Si algún álbum tuviera un "ArtistId" que no existe en "Artist", igual aparecería, con None en la columna "Artista".

In [49]:
%%sql
SELECT 
    ar.Name AS Artista,
    a.Title AS Album
FROM Artist AS ar
RIGHT JOIN Album AS a ON a.ArtistId = ar.ArtistId
ORDER BY a.Title
LIMIT 5;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Artista,Album
Metallica,...And Justice For All
Scorpions,20th Century Masters - The Millennium Collection: The Best of Scorpions
Aaron Copland & London Symphony Orchestra,"A Copland Celebration, Vol. I"
Iron Maiden,A Matter of Life and Death
Iron Maiden,A Real Dead One


#### Filtrar en ON del Join
La condicion de aplica antes de unir las tablas, sobre que filas de la tabla derecha va a traer, pero sin eliminar filas de la tabla izquierda eso es lo que pasa con "LEFT JOIN" y en caso de "RIGTH JOIN" lo mismo pero el filtro se aplica en la tabla izquierda y la tabla derecha no se eliminan filas.

Con "INNER JOIN" filtrar en ON o WHERE da normalmente el mismo resultado. La diferencia se nota cuando se utiliza "LEFT JOIN" o "RIGHT JOIN".

En el siguiente ejemplo la condicion "a.Title != 'Unplugged'" se aplica durante el join. Lo que quiere decir es que no "Unplugged" no aparecera emparejado con el artista, sin embargo el artista sigue apareciendo y donde deberia estar "Unplugged" aparecera como dato Null o None en este caso.

In [51]:
%%sql
SELECT 
    ar.Name AS Artista,
    a.Title AS Album
FROM Artist AS ar
LEFT JOIN Album AS a 
    ON a.ArtistId = ar.ArtistId
    AND a.Title != 'Unplugged'
ORDER BY ar.Name
LIMIT 5;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Artista,Album
A Cor Do Som,None
AC/DC,For Those About To Rock We Salute You
AC/DC,Let There Be Rock
Aaron Copland & London Symphony Orchestra,"A Copland Celebration, Vol. I"
Aaron Goldberg,Worlds


#### Filtrar en WHERE del Join

En el siguiente ejemplo se hace un "LEFT JOIN" completo, trae toda la informacion incluso los None. Despues, "WHERE" descarta cualquier fila donde el album se llame "Unplugged".

Como "WHERE" se ejecuta despues del join, si no se agrega "OR a.Title IS NULL" se perdera por completo a los artistas que no tengan ningun album. Ya que en "SQL" los datos nulos requieren tratamiento especial. Por lo que para no perder a los artistas que tengan datos null y mantener los artistas sin albumes.

In [52]:
%%sql
SELECT 
    ar.Name AS Artista,
    a.Title AS Album
FROM Artist AS ar
LEFT JOIN Album AS a 
    ON a.ArtistId = ar.ArtistId
WHERE a.Title != 'Unplugged'
   OR a.Title IS NULL
ORDER BY ar.Name
LIMIT 5;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Artista,Album
A Cor Do Som,None
AC/DC,For Those About To Rock We Salute You
AC/DC,Let There Be Rock
Aaron Copland & London Symphony Orchestra,"A Copland Celebration, Vol. I"
Aaron Goldberg,Worlds


Este ultimo ejemplo lista todos los artistas con la cantidad de albumes que tiene, incluyendo a los arsitas sin album, ordenando de mayor a menor.

Se utiliza "COUNT(a.AlbumId)" ya que de esta manera se ignoran los datos null, ya que al utilizar "COUNT(*)" se contarian tambien los datos nulos. 

In [53]:
%%sql
SELECT 
    ar.Name AS Artista,
    COUNT(a.AlbumId) AS Cantidad_Albumes
FROM Artist AS ar
LEFT JOIN Album AS a ON a.ArtistId = ar.ArtistId
GROUP BY ar.Name
ORDER BY Cantidad_Albumes DESC
LIMIT 5;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Artista,Cantidad_Albumes
Iron Maiden,21
Led Zeppelin,14
Deep Purple,11
U2,10
Metallica,10


#### Full Join
Devuelve filas no coincidentes en ambas tablas y lo que si coincide, en otras palabras devuelve todo de ambas tablas. Se usa comunmente junto con agregaciones para comprender la cantidad de superposicion entre dos tablas. Es para ver que tan bien se relacionan dos tablas.

In [54]:
%%sql
SELECT 
    COUNT(CASE WHEN ar.ArtistId IS NOT NULL AND a.ArtistId IS NULL 
               THEN ar.ArtistId ELSE NULL END) AS solo_artistas_sin_album,
    COUNT(CASE WHEN ar.ArtistId IS NOT NULL AND a.ArtistId IS NOT NULL 
               THEN ar.ArtistId ELSE NULL END) AS en_ambas_tablas,
    COUNT(CASE WHEN ar.ArtistId IS NULL AND a.ArtistId IS NOT NULL 
               THEN a.ArtistId ELSE NULL END) AS solo_albumes_sin_artista
FROM Artist AS ar
FULL JOIN Album AS a
    ON ar.ArtistId = a.ArtistId;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

solo_artistas_sin_album,en_ambas_tablas,solo_albumes_sin_artista
71,347,0


#### Operador Unior

Unior permite apilar un conjunto de datos encima de otro. En otras palabra permite escribir dos consultas "SELECT" separadas y los resultados se muestren en la misma tabla que los resultados de la otra.

Aunque "UNION" solo agrega valores distintos. Se puede utilizar "UNION ALL" para agregar todos los valores de la segunda tabla aunque sean duplicados de la primera.

SQL tiene reglas estrictas para agregar datos: 
- Ambas tablas deben tener el mismo numero de columnas.
- Las columnas deben tener los mismos tipos de datos en el mismo orden que la primera tabla.


In [59]:
%%sql
SELECT 
    a.Title AS Album,
    COUNT(t.TrackId) AS canciones_largas
FROM Album AS a
LEFT JOIN Track AS t 
    ON t.AlbumId = a.AlbumId
    AND t.Milliseconds > 240000
GROUP BY a.Title
ORDER BY canciones_largas DESC
LIMIT 5;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Album,canciones_largas
Greatest Hits,37
"Lost, Season 3",26
"The Office, Season 3",25
"Lost, Season 1",25
"Lost, Season 2",24


In [60]:
%%sql
SELECT 
    a.Title AS Album,
    COUNT(t.TrackId) AS canciones_largas
FROM Album AS a
LEFT JOIN Track AS t 
    ON t.AlbumId = a.AlbumId
WHERE t.Milliseconds > 240000
GROUP BY a.Title
ORDER BY canciones_largas DESC
LIMIT 5;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Album,canciones_largas
Greatest Hits,37
"Lost, Season 3",26
"The Office, Season 3",25
"Lost, Season 1",25
"Lost, Season 2",24


La diferencia entre los codigos es que en el primero se utiliza filtrado en el ON lo que deja como resultados a albumes aunque no tengan canciones que superen la comparacion de "240000" milisegundos, por lo que igualmente se muestra el album aunque no tenga canciones que cumplan el filtro.

Y en el segundo codigo se realiza el filtro en el "WHERE" este no muestra los albumes que no tengan canciones que cumplen con el filtro.

##### Uniones con multiples claves

A veces conviene unir dos tablas usando más de una columna en el `ON`, por dos razones:

- **Precisión:** si dos tablas comparten varias columnas en común, unir por todas reduce el riesgo de emparejamientos ambiguos o incorrectos (útil cuando una sola columna no garantiza unicidad).
- **Rendimiento:** SQL usa índices para acelerar consultas. Unir por varias columnas puede hacer la consulta más rápida aunque no cambie el resultado, ya que la base de datos puede aprovechar mejor sus índices internos. Este efecto es mínimo en datasets pequeños (como Chinook) y se nota más en tablas con millones de filas.

In [61]:
%%sql
SELECT 
    i.InvoiceId,
    i.CustomerId,
    il.TrackId,
    il.UnitPrice
FROM Invoice AS i
LEFT JOIN InvoiceLine AS il
    ON i.InvoiceId = il.InvoiceId

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

InvoiceId,CustomerId,TrackId,UnitPrice
98,1,3247,1.99
98,1,3248,1.99
121,1,447,0.99
121,1,449,0.99
121,1,451,0.99
121,1,453,0.99
143,1,1153,0.99
143,1,1157,0.99
143,1,1161,0.99
143,1,1165,0.99


Esta técnica es más relevante en datasets menos normalizados, donde una sola columna no basta para asegurar una relación única (ej. unir por `nombre` + `fecha de nacimiento` en vez de solo `nombre`).

##### Uniones automaticas

Puede ser util unir una tabla consigo misma, es una tecnica comun para comparar filas de una tabla entre si, no con otras tablas distintas.

In [62]:
%%sql
SELECT DISTINCT 
    e1.FirstName || ' ' || e1.LastName AS Empleado_1,
    e2.FirstName || ' ' || e2.LastName AS Empleado_2,
    e1.City
FROM Employee AS e1
JOIN Employee AS e2
    ON e1.City = e2.City
   AND e1.EmployeeId < e2.EmployeeId
ORDER BY e1.City;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Empleado_1,Empleado_2,City
Nancy Edwards,Steve Johnson,Calgary
Nancy Edwards,Michael Mitchell,Calgary
Nancy Edwards,Margaret Park,Calgary
Nancy Edwards,Jane Peacock,Calgary
Jane Peacock,Steve Johnson,Calgary
Jane Peacock,Michael Mitchell,Calgary
Jane Peacock,Margaret Park,Calgary
Margaret Park,Steve Johnson,Calgary
Margaret Park,Michael Mitchell,Calgary
Steve Johnson,Michael Mitchell,Calgary


In [63]:
%%sql
SELECT 
    e.FirstName || ' ' || e.LastName AS Empleado,
    jefe.FirstName || ' ' || jefe.LastName AS Jefe
FROM Employee AS e
LEFT JOIN Employee AS jefe
    ON e.ReportsTo = jefe.EmployeeId
ORDER BY Jefe;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Empleado,Jefe
Andrew Adams,None
Nancy Edwards,Andrew Adams
Michael Mitchell,Andrew Adams
Robert King,Michael Mitchell
Laura Callahan,Michael Mitchell
Jane Peacock,Nancy Edwards
Margaret Park,Nancy Edwards
Steve Johnson,Nancy Edwards


Cuando usar "SELF JOIN":
- Comparar filas de la misma tabla entre si.
- Relaciones jerarquicas dentro de una misma tabla.
- Entrontrar pares o relaciones temporales dentro del mismo conjunto de datos.

# Conclusión

En esta práctica cubrí SQL intermedio trabajando con la base de datos Chinook: 
funciones de agregación (`COUNT`, `SUM`, `MIN`/`MAX`, `AVG`), agrupación de datos 
con `GROUP BY` (incluyendo agrupación por número de columna y su combinación con 
`ORDER BY`), filtrado de grupos con `HAVING` y su diferencia clave con `WHERE` 
(el orden real de ejecución de las cláusulas SQL: `FROM` → `WHERE` → `GROUP BY` → 
`HAVING` → `SELECT` → `ORDER BY`), lógica condicional con `CASE`, y valores únicos 
con `DISTINCT`.

También profundicé en JOINs: `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN` y `FULL JOIN` 
con sus diferencias visuales, la distinción importante entre filtrar en la cláusula 
`ON` versus en `WHERE` (especialmente relevante con `LEFT JOIN`, donde puede cambiar 
qué filas se conservan), uniones con múltiples claves, uniones de 3 o más tablas, 
self joins para comparar filas de una misma tabla entre sí, y el operador `UNION` 
para apilar resultados de distintas consultas.

Este notebook se enfocó en consolidar la teoría y sintaxis de cada concepto con 
ejemplos aplicados sobre Chinook. La práctica dirigida con preguntas de negocio y 
ejercicios progresivos de joins queda pendiente para un notebook futuro, trabajando 
sobre una base de datos más grande — esto permitirá aplicar estos conceptos a 
consultas más complejas y realistas.